# Interview Prep: 14 - System Design Python

For Senior and Staff roles, the interview often shifts from "how to write a function" to "how to design a system". This notebook covers the architectural trade-offs specific to Python-based microservices.

---

## 1. gRPC vs. REST

### **Interview Question**: "When would you use gRPC over REST for service-to-service communication?"

**Answer**:
- **gRPC**: Uses HTTP/2 and Protocol Buffers (binary). It is much faster, has lower overhead, and provides strong typing out of the box. Best for **internal** microservices.
- **REST**: Uses HTTP/1.1 and JSON (text). It is human-readable, browser-native, and has a massive ecosystem. Best for **public-facing** APIs.

---

## 2. Resiliency: The Circuit Breaker

### **Interview Question**: "What is a Circuit Breaker, and why is it important in a distributed system?"

**Answer**: A circuit breaker prevents a failing service from causing a cascading failure across the whole system. When a service fails repeatedly, the breaker "opens" and immediately returns an error (or fallback) for all requests, giving the failing service time to recover.

#### **Key States**:
- **Closed**: Requests pass through normally.
- **Open**: Requests fail immediately.
- **Half-Open**: A few requests are allowed through to see if the service has recovered.

---

## 3. The Saga Pattern (Distributed Transactions)

### **Interview Question**: "How do you handle a transaction that spans three different microservices?"

**Answer**: You use the **Saga Pattern**. Instead of one large transaction, each service performs its local transaction and publishes an event. If one step fails, the previous services must run **Compensating Transactions** to undo their changes.

---

## 4. Coding Challenge: Implementing Retries with Tenacity

**Task**: Use the `tenacity` patterns to implement a robust retry logic with exponential backoff and jitter for an unstable API call.


In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import requests
import random

# Scenario: Unstable external service
class ExternalServiceError(Exception): pass

@retry(
    stop=stop_after_attempt(5), 
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(ExternalServiceError)
)
def call_unstable_api():
    if random.random() < 0.7:  # 70% failure rate
        print("API Call Failed... Retrying")
        raise ExternalServiceError("Server is down!")
    print("API Call Succeeded!")
    return "Success"

try:
    call_unstable_api()
except Exception as e:
    print(f"Final Failure: {e}")

## 5. Observability: Distributed Tracing

### **Interview Question**: "How do you trace a single request through 10 different services?"

**Answer**: You use a **Correlation ID** (or Trace ID). The API Gateway generates a unique ID for every incoming request and passes it in the headers (e.g., `X-Correlation-ID`) to every downstream service. Tools like **OpenTelemetry** and **Jaeger** automate this process.

---